# Notebook 02 — Normalization Walkthrough and Enriched Dataset Export

This notebook shows how preprocessing settings change text and then writes an
enriched `articles.csv`-style file with both normalized views.

Reader guide:

- I first use short synthetic examples for quick sanity checks.
- I then process the full raw dataset from `data/raw/articles.csv`.
- I write a new enriched file with both views in `data/processed/articles.csv`.
- I include a small bag-of-words matrix preview to show matrix structure.

## Setup

Add the `src/` directory to the Python path so that project modules are importable without installing the package.

In [1]:
import sys
from pathlib import Path

project_root_path = Path.cwd().parent
if str(project_root_path / "src") not in sys.path:
    sys.path.insert(0, str(project_root_path / "src"))

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from core.data_io import ArticleDataset
from preprocessing import TextNormalizer, TextPreprocessor

## Synthetic sanity check

These samples intentionally include:

- HTML tags
- URLs and email-like tokens
- heavy punctuation and reference-like IDs

This makes it easy to inspect what each normalization view keeps or removes.

In [3]:
sample_texts = [
    "<p>Engine failure at STATION-44!!! urgent alert</p>",
    "Visit https://bad-shop.example now and mail me@bad-shop.example",
    "%%%% Buy now #### REF-8821 ####",
]

normalizer = TextNormalizer()
# Build both views from identical input for direct comparison.
bundle = normalizer.normalize_for_both_tasks(sample_texts)

normalization_examples_data_frame = pd.DataFrame(
    {
        "raw": sample_texts,
        "clustering_view": bundle.clustering_texts,
        "anomaly_view": bundle.anomaly_texts,
    }
)
normalization_examples_data_frame

,raw,clustering_view,anomaly_view
0,<p>Engine failure at STATION-44!!! urgent aler...,engine failure station urgent alert,p engine failure station urgent alert p < > - ...
1,Visit https://bad-shop.example now and mail me...,visit mail,visit https bad shop example mail bad shop exa...
2,%%%% Buy now #### REF-8821 ####,buy ref,buy ref % % % % # # # # - # # # #


## Build enriched dataset and write new articles.csv

In this section I process the full assignment dataset and create a new CSV that
keeps the original columns plus both normalized views.

I write this to `data/processed/articles.csv` so the raw source in `data/raw/`
remains untouched.

In [4]:
project_root_path = Path.cwd().parent
processed_data_directory_path = project_root_path / "data" / "processed"
processed_data_directory_path.mkdir(parents=True, exist_ok=True)

input_articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
enriched_articles_csv_path = processed_data_directory_path / "articles.csv"

articles_data_frame = ArticleDataset(input_csv_path=input_articles_csv_path).load_articles()
raw_texts = articles_data_frame["text"].tolist()
full_bundle = normalizer.normalize_for_both_tasks(raw_texts)

enriched_articles_data_frame = articles_data_frame.copy()
enriched_articles_data_frame["clustering_view"] = full_bundle.clustering_texts
enriched_articles_data_frame["anomaly_view"] = full_bundle.anomaly_texts

# Write a strict CSV with an explicit header and column order.
# This helps external data viewers detect column names correctly.
export_column_names = ["doc_id", "text", "clustering_view", "anomaly_view"]
enriched_articles_data_frame.to_csv(
    enriched_articles_csv_path,
    columns=export_column_names,
    index=False,
    header=True,
    encoding="utf-8",
    lineterminator="\n",
)

enriched_articles_data_frame.head(10)

,doc_id,text,clustering_view,anomaly_view
0,DOC_00001,"When I first saw this, I thought for a second ...",saw thought second headline star plier srb rec...,saw thought second headline star pliers srb re...
1,DOC_00002,The difficulties of a high Isp OTV include: Lo...,difficulty high isp otv include long transfer ...,difficulties high isp otv include long transfe...
2,DOC_00003,"Forwarded from Neal Ausman, Galileo Mission Di...",forwarded neal ausman galileo mission director...,forwarded neal ausman galileo mission director...
3,DOC_00004,Sjogren's syndrome has been known to induce dr...,sjogren syndrome known induce dryness vaginal ...,sjogren syndrome known induce dryness vaginal ...
4,DOC_00005,"Yes, I want to concentrate on other developmen...",yes want concentrate development issue created...,yes want concentrate development issues create...
5,DOC_00006,I have a 86 chevy sprint with a/c and 4doors. ...,chevy sprint c odometer turned sensor light st...,chevy sprint c doors odometer turned k sensor ...
6,DOC_00007,SEI Level 5 (the highest level -- the SEI stan...,sei level highest level sei stand software eng...,sei level highest level sei stands software en...
7,DOC_00008,Assuming one has been cultured as having a thr...,assuming cultured having throat laden neiseria...,assuming cultured having throat laden neiseria...
8,DOC_00009,I've been asking myself this same question for...,asking question past year share magistic answe...,asking question past year share magistic answe...
9,DOC_00010,Living things maintain small electric fields t...,living thing maintain small electric field enh...,living things maintain small electric fields e...


## Export validation (header check)

This quick readback confirms the written file starts with the expected header
columns. If your data viewer still shows `C1`, `C2`, ... then switch its option
for "First row is header" and reopen the file.

In [5]:
exported_articles_data_frame = pd.read_csv(enriched_articles_csv_path)
print(exported_articles_data_frame.columns.tolist())
exported_articles_data_frame.head(5)

['doc_id', 'text', 'clustering_view', 'anomaly_view']


,doc_id,text,clustering_view,anomaly_view
0,DOC_00001,"When I first saw this, I thought for a second ...",saw thought second headline star plier srb rec...,saw thought second headline star pliers srb re...
1,DOC_00002,The difficulties of a high Isp OTV include: Lo...,difficulty high isp otv include long transfer ...,difficulties high isp otv include long transfe...
2,DOC_00003,"Forwarded from Neal Ausman, Galileo Mission Di...",forwarded neal ausman galileo mission director...,forwarded neal ausman galileo mission director...
3,DOC_00004,Sjogren's syndrome has been known to induce dr...,sjogren syndrome known induce dryness vaginal ...,sjogren syndrome known induce dryness vaginal ...
4,DOC_00005,"Yes, I want to concentrate on other developmen...",yes want concentrate development issue created...,yes want concentrate development issues create...


## Bag-of-words matrix preview (small slice)

The complete matrix is very high-dimensional, so I show only:

- a small number of documents,
- the globally most frequent terms.

In [6]:
bow_preprocessor = TextPreprocessor(
    vectorization_model_name="bow",
    min_document_frequency=2,
    max_document_frequency=0.95,
    ngram_range=(1, 1),
)

bow_sparse_matrix = bow_preprocessor.fit_transform(full_bundle.clustering_texts)
feature_names = bow_preprocessor.get_feature_names()

term_frequency_array = np.asarray(bow_sparse_matrix.sum(axis=0)).ravel()
sorted_term_indices = np.argsort(term_frequency_array)[::-1]

preview_document_count = 12
preview_term_count = 20
preview_term_indices = sorted_term_indices[:preview_term_count]
preview_term_names = [feature_names[index] for index in preview_term_indices]

preview_dense_matrix = bow_sparse_matrix[:preview_document_count, preview_term_indices].toarray().astype(int)
preview_document_ids = enriched_articles_data_frame["doc_id"].tolist()[:preview_document_count]

bow_preview_data_frame = pd.DataFrame(
    preview_dense_matrix,
    columns=preview_term_names,
    index=preview_document_ids,
)
bow_preview_data_frame.index.name = "doc_id"
bow_preview_data_frame

,pickup,deal,like,offer,time,fast,sale,just,know,bonu,year,think,car,good,cash,new,doe,game,did,limited
doc_id,,,,,,,,,,,,,,,,,,,,
DOC_00001,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
DOC_00002,0,0,1,0,2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
DOC_00003,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
DOC_00004,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
DOC_00005,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
DOC_00006,0,0,0,0,0,0,0,0,2,0,0,0,1,0,0,0,0,0,1,0
DOC_00007,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,1,0,0,1,0
DOC_00008,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
DOC_00009,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0


### Files produced by this notebook

- `data/processed/articles.csv` (original `doc_id` + `text` + `clustering_view` + `anomaly_view`)
